# 03 — Label Construction
## SustAInable | Neighborhood Heat Illness Risk Prediction

---

**Purpose:** Builds the outcome variable (Y label) that the XGBoost model will learn to predict.

For each ZCTA and each historical heat event, we answer one question:

> *Did this ZIP code experience elevated heat illness during this event?*

| Label | Definition |
|---|---|
| **1 — Elevated** | ZCTA experienced ≥ 2× its 3-year baseline ED visit rate per 10,000 residents during a heat event |
| **0 — Not Elevated** | ZCTA did not experience elevated heat illness during the event |

**Inputs:**
- `data/processed/sustainable_features_merged.csv` ← from Notebook 02
- NSSP ED visit data *(restricted — requires BioSense DUA)*
- CDC WONDER mortality data *(public — used as proxy if NSSP unavailable)*
- NOAA CDO historical temperature records *(public)*

**Output:** `data/processed/sustainable_labeled.csv`

**Author:** Nico Meyering, MPA | Equitech Futures Civic Tech Institute, CTI 2026

---

> ⚠️ **A note on data access:** The primary outcome variable requires NSSP syndromic surveillance data, which is restricted. This notebook is built to run in two modes:
> - **Full mode** — NSSP data available; real ED visit labels constructed
> - **Proxy mode** — NSSP unavailable; CDC WONDER mortality + synthetic demonstration labels used so model development can proceed while DUA is pending

The notebook auto-detects which mode to use based on what files are present.

## Step 0: Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Imports complete.')
print(f'pandas {pd.__version__} | numpy {np.__version__}')

## Step 1: File Paths and Mode Detection

This cell checks which data files exist and automatically selects **Full mode** (NSSP available) or **Proxy mode** (NSSP unavailable).

In [ ]:
DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
OUTPUTS        = Path("../outputs")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# ── Feature matrix from Notebook 02 ───────────────────────────────────────────
FEATURES_PATH = DATA_PROCESSED / "sustainable_features_merged.csv"

# ── NSSP candidates (restricted data) ─────────────────────────────────────────
NSSP_CANDIDATES = [
    "nssp_heat_ed_visits.csv",
    "nssp_heat_illness.csv",
    "nssp_t67_zcta.csv",
    "biosense_heat_ed.csv",
    "heat_ed_visits_zcta.csv",
    "nssp_ed_zcta.csv",
]

# ── CDC WONDER candidates (public proxy) ──────────────────────────────────────
WONDER_CANDIDATES = [
    "cdc_wonder_heat_deaths.csv",
    "wonder_heat_mortality.csv",
    "cdc_wonder_x30.csv",
    "heat_mortality_county.csv",
    "wonder_compressed_mortality.csv",
    "cdc_wonder.csv",
]

# ── NOAA CDO historical weather candidates ────────────────────────────────────
NOAA_CANDIDATES = [
    "noaa_cdo_daily.csv",
    "noaa_ghcnd_summer.csv",
    "noaa_temperature.csv",
    "noaa_daily_summaries.csv",
    "ghcnd_daily.csv",
    "noaa_cdo.csv",
]

def find_first(candidates, dirs):
    for d in dirs:
        for name in candidates:
            p = d / name
            if p.exists():
                return p
    return None

search_dirs = [DATA_RAW, DATA_PROCESSED]
features_available = FEATURES_PATH.exists()
nssp_path   = find_first(NSSP_CANDIDATES, search_dirs)
wonder_path = find_first(WONDER_CANDIDATES, search_dirs)
noaa_path   = find_first(NOAA_CANDIDATES, search_dirs)

print("=" * 55)
print("FILE DETECTION REPORT")
print("=" * 55)
print(f"Feature matrix (Notebook 02): {'✅  ' + FEATURES_PATH.name if features_available else '⚠️   NOT FOUND'}")
print(f"NSSP ED visit data:           {'✅  ' + nssp_path.name if nssp_path else '⚠️   Not found (restricted)'}")
print(f"CDC WONDER mortality:         {'✅  ' + wonder_path.name if wonder_path else 'ℹ️   Not found (optional proxy)'}")
print(f"NOAA CDO weather:             {'✅  ' + noaa_path.name if noaa_path else 'ℹ️   Not found (optional)'}")

# ── Mode selection ─────────────────────────────────────────────────────────────
if nssp_path:
    MODE = "full"
    print("\n🟢  MODE: FULL — NSSP data available. Real ED visit labels will be constructed.")
else:
    MODE = "proxy"
    print("\n🟡  MODE: PROXY — NSSP data not available.")
    print("    Labels will use CDC WONDER mortality + structurally-derived proxy.")
    print("    This is expected during early development. Full labels require a")
    print("    BioSense Platform DUA: https://www.cdc.gov/nssp/php/onboarding/index.html")

if not features_available:
    print("\n🔴  BLOCKER: Feature matrix not found.")
    print(f"    Expected: {FEATURES_PATH}")
    print("    Run Notebook 02 first and save its output before continuing.")

## Step 2: Load Feature Matrix

Loads the output of Notebook 02. Every ZCTA in this file will receive a label.

In [ ]:
features = None

if FEATURES_PATH.exists():
    features = pd.read_csv(FEATURES_PATH, dtype={'ZCTA': str})
    features['ZCTA'] = features['ZCTA'].str.zfill(5)
    print(f"✅ Feature matrix loaded.")
    print(f"   ZCTAs:   {len(features):,}")
    print(f"   Columns: {features.shape[1]}")
    display(features[['ZCTA'] + [c for c in features.columns if c != 'ZCTA'][:5]].head(3))
else:
    print("🔴 Feature matrix not found. Run Notebook 02 first.")
    print(f"   Expected path: {FEATURES_PATH}")

## Step 3: Define Historical Heat Events

A heat event is defined as any period where NOAA HeatRisk reached Level 3 or higher for at least 2 consecutive days.

**If you have NOAA CDO data:** This cell builds events from actual temperature records.

**If you don't have NOAA CDO data yet:** A validated reference list of major US heat events (2010–2024) is built in, drawn from published NOAA and CDC records. This is sufficient for Phase 2 development.

> Think of heat events as the "game days" — each event is one observation per ZCTA. A ZCTA that saw elevated illness on 3 separate events contributes 3 labeled rows to the training set.

In [ ]:
# ── Reference heat events (2010–2024) ─────────────────────────────────────────
# Drawn from NOAA Storm Data, CDC Heat Morbidity Surveillance, and published literature.
# Format: (event_id, label, start_date, end_date, primary_regions)
REFERENCE_HEAT_EVENTS = [
    ("HE_2011_TX",   "2011 Texas Heat Emergency",          "2011-07-01", "2011-08-31", "South, Central"),
    ("HE_2012_MW",   "2012 Midwest/Central Heat",          "2012-06-28", "2012-07-08", "Midwest, East"),
    ("HE_2013_W",    "2013 Western Heat",                  "2013-06-28", "2013-07-02", "West"),
    ("HE_2015_PNW",  "2015 Pacific Northwest Heat",        "2015-06-26", "2015-07-01", "Pacific Northwest"),
    ("HE_2016_SW",   "2016 Southwest Heat",                "2016-06-19", "2016-06-22", "Southwest"),
    ("HE_2018_NE",   "2018 Northeast Heat",                "2018-07-01", "2018-07-06", "Northeast"),
    ("HE_2019_MW",   "2019 Midwest Heat",                  "2019-07-16", "2019-07-20", "Midwest"),
    ("HE_2020_CA",   "2020 California Heat Emergency",     "2020-08-14", "2020-09-07", "West"),
    ("HE_2021_PNW",  "2021 Pacific Northwest Heat Dome",   "2021-06-26", "2021-07-01", "Pacific Northwest"),
    ("HE_2022_SW",   "2022 Southwest / Texas Heat",        "2022-06-09", "2022-06-16", "South, Southwest"),
    ("HE_2022_NE",   "2022 Northeast / Mid-Atlantic Heat", "2022-07-18", "2022-07-22", "Northeast"),
    ("HE_2023_TX",   "2023 Texas Record Heat",             "2023-06-15", "2023-07-15", "South, Central"),
    ("HE_2023_SW",   "2023 Southwest Heat",                "2023-07-09", "2023-07-17", "Southwest"),
    ("HE_2024_PHX",  "2024 Phoenix Multi-Week Heat",       "2024-06-15", "2024-07-10", "Southwest"),
    ("HE_2024_NE",   "2024 Northeast / Mid-Atlantic Heat", "2024-06-17", "2024-06-24", "Northeast"),
]

events_df = pd.DataFrame(
    REFERENCE_HEAT_EVENTS,
    columns=["event_id", "event_label", "start_date", "end_date", "primary_regions"]
)
events_df["start_date"] = pd.to_datetime(events_df["start_date"])
events_df["end_date"]   = pd.to_datetime(events_df["end_date"])
events_df["duration_days"] = (events_df["end_date"] - events_df["start_date"]).dt.days + 1
events_df["year"] = events_df["start_date"].dt.year

# ── Split into training and holdout sets ──────────────────────────────────────
# Training: 2010–2021 | Holdout: 2022–2024 (as defined in model design)
train_events   = events_df[events_df["year"] <= 2021].copy()
holdout_events = events_df[events_df["year"] >= 2022].copy()

print("=" * 55)
print("HEAT EVENT REGISTRY")
print("=" * 55)
print(f"Total events:   {len(events_df)}")
print(f"Training set:   {len(train_events)} events (2010–2021)")
print(f"Holdout set:    {len(holdout_events)} events (2022–2024)")
print()
print("Training events:")
display(train_events[["event_id","event_label","start_date","end_date","duration_days"]].reset_index(drop=True))
print()
print("Holdout events (stress tests):")
display(holdout_events[["event_id","event_label","start_date","end_date","duration_days"]].reset_index(drop=True))

## Step 4A: Build Labels — Full Mode (NSSP Data Available)

*This cell only runs if NSSP data was detected in Step 1.*

NSSP data contains ED visit counts by ZCTA by day for ICD-10 codes T67.x (heat-related illness). We calculate a 3-year rolling baseline per ZCTA, then flag any event-period where visits exceeded 2× that baseline.

In [ ]:
nssp_labels = None

if MODE == "full" and nssp_path is not None:
    print(f"Loading NSSP data from: {nssp_path.name}")
    nssp_raw = pd.read_csv(nssp_path, dtype=str)

    print(f"Shape: {nssp_raw.shape}")
    print(f"Columns: {list(nssp_raw.columns)}")
    display(nssp_raw.head(3))

    # ── Identify key columns ───────────────────────────────────────────────────
    zcta_col_candidates  = ['ZCTA', 'ZCTA5', 'zip_code', 'zipcode', 'ZIP']
    date_col_candidates  = ['date', 'Date', 'visit_date', 'Date_of_visit', 'DATE']
    count_col_candidates = ['visit_count', 'count', 'ed_visits', 'Count',
                            'num_visits', 'total_visits', 'visits']
    pop_col_candidates   = ['population', 'pop', 'Population', 'total_pop']

    zcta_col  = next((c for c in zcta_col_candidates  if c in nssp_raw.columns), None)
    date_col  = next((c for c in date_col_candidates  if c in nssp_raw.columns), None)
    count_col = next((c for c in count_col_candidates if c in nssp_raw.columns), None)
    pop_col   = next((c for c in pop_col_candidates   if c in nssp_raw.columns), None)

    print(f"\nColumn mapping:")
    print(f"  ZCTA:       {zcta_col}")
    print(f"  Date:       {date_col}")
    print(f"  Visit count:{count_col}")
    print(f"  Population: {pop_col or 'Not found — will join from features'}")

    if not all([zcta_col, date_col, count_col]):
        print("\n⚠️  Could not auto-detect required columns.")
        print("   Set them manually:")
        print("     zcta_col  = 'your_zcta_column_name'")
        print("     date_col  = 'your_date_column_name'")
        print("     count_col = 'your_count_column_name'")
    else:
        nssp = nssp_raw[[zcta_col, date_col, count_col]].copy()
        nssp.columns = ['ZCTA', 'date', 'ed_visits']
        nssp['ZCTA']     = nssp['ZCTA'].astype(str).str.zfill(5)
        nssp['date']     = pd.to_datetime(nssp['date'])
        nssp['ed_visits'] = pd.to_numeric(nssp['ed_visits'], errors='coerce').fillna(0)

        # ── Join population from features ──────────────────────────────────────
        pop_candidates_feat = [c for c in features.columns
                               if 'pop' in c.lower() or 'population' in c.lower()]
        if features is not None and pop_candidates_feat:
            pop_col_feat = pop_candidates_feat[0]
            pop_lookup = features[['ZCTA', pop_col_feat]].rename(
                columns={pop_col_feat: 'population'}
            )
            nssp = nssp.merge(pop_lookup, on='ZCTA', how='left')
        else:
            nssp['population'] = 10000  # placeholder if no pop data yet

        # ── 3-year rolling baseline per ZCTA ──────────────────────────────────
        nssp['year']  = nssp['date'].dt.year
        nssp['month'] = nssp['date'].dt.month
        nssp_summer   = nssp[nssp['month'].isin([6, 7, 8, 9])].copy()

        baseline = (
            nssp_summer
            .groupby(['ZCTA', 'year'])['ed_visits']
            .sum()
            .reset_index()
            .rename(columns={'ed_visits': 'annual_visits'})
        )
        baseline['visits_per_10k'] = (
            baseline['annual_visits'] / nssp_summer
            .groupby(['ZCTA', 'year'])['population']
            .first()
            .values
        ) * 10000

        baseline_3yr = (
            baseline
            .groupby('ZCTA')
            .apply(lambda x: x.nlargest(3, 'year')['visits_per_10k'].mean())
            .reset_index()
            .rename(columns={0: 'baseline_rate_per_10k'})
        )

        # ── Label construction per event ──────────────────────────────────────
        label_rows = []
        for _, event in events_df.iterrows():
            event_visits = nssp[
                (nssp['date'] >= event['start_date']) &
                (nssp['date'] <= event['end_date'])
            ].groupby('ZCTA').agg(
                event_visits=('ed_visits', 'sum'),
                population=('population', 'first')
            ).reset_index()

            event_visits['event_rate_per_10k'] = (
                event_visits['event_visits'] / event_visits['population']
            ) * 10000
            event_visits = event_visits.merge(baseline_3yr, on='ZCTA', how='left')
            event_visits['label'] = (
                event_visits['event_rate_per_10k'] >=
                2 * event_visits['baseline_rate_per_10k']
            ).astype(int)
            event_visits['event_id'] = event['event_id']
            label_rows.append(event_visits)

        nssp_labels = pd.concat(label_rows, ignore_index=True)
        print(f"\n✅ NSSP labels constructed.")
        print(f"   Total ZCTA-event rows: {len(nssp_labels):,}")
        print(f"   Class distribution:")
        print(nssp_labels['label'].value_counts().to_string())
        display(nssp_labels.head(5))
else:
    print("ℹ️  Skipping Full Mode (NSSP not available).")
    print("   Proceeding to Proxy Mode in Step 4B.")

## Step 4B: Build Labels — Proxy Mode (NSSP Unavailable)

*This cell runs when NSSP data is not available — which is the expected state during early development.*

**What the proxy does:**

Since NSSP requires a DUA, we construct a structurally-derived proxy label using:
1. **CDC WONDER** mortality data (public, county-level) — if available
2. **Structural vulnerability features** from the merged feature matrix — ZCTAs in the top quartile of composite vulnerability are more likely to have experienced elevated illness; this is used to generate statistically plausible demonstration labels

> ⚠️ **Important:** Proxy labels are for model development and pipeline testing only. They are NOT suitable for real-world deployment decisions. The label section of `README.md` and any publications must clearly note that NSSP data is required for production labels. This notebook will add a flag column `label_is_proxy = 1` to every proxy-labeled row so the distinction is always traceable.

In [ ]:
proxy_labels = None

if MODE == "proxy" and features is not None:
    print("🟡 Building proxy labels from structural vulnerability features.")
    print()

    # ── Load CDC WONDER if available ──────────────────────────────────────────
    wonder_df = None
    if wonder_path:
        try:
            wonder_raw = pd.read_csv(wonder_path, dtype=str, encoding='latin-1')
            # WONDER exports often have note rows at the bottom — strip them
            wonder_raw = wonder_raw[wonder_raw.iloc[:, 0].notna()]
            # Try to find county FIPS and death count columns
            fips_candidates  = ['County Code', 'FIPS', 'county_code', 'CountyCode']
            death_candidates = ['Deaths', 'death_count', 'Count', 'deaths']
            fips_col  = next((c for c in fips_candidates  if c in wonder_raw.columns), None)
            death_col = next((c for c in death_candidates if c in wonder_raw.columns), None)
            if fips_col and death_col:
                wonder_df = wonder_raw[[fips_col, death_col]].copy()
                wonder_df.columns = ['county_fips', 'heat_deaths']
                wonder_df['county_fips'] = wonder_df['county_fips'].astype(str).str.zfill(5)
                wonder_df['heat_deaths'] = pd.to_numeric(
                    wonder_df['heat_deaths'], errors='coerce'
                ).fillna(0)
                print(f"✅ CDC WONDER loaded: {len(wonder_df):,} counties")
            else:
                print(f"ℹ️  CDC WONDER found but columns unrecognized: {list(wonder_raw.columns)}")
        except Exception as e:
            print(f"ℹ️  CDC WONDER load error: {e}")
    else:
        print("ℹ️  CDC WONDER not found — proxy will use structural features only.")

    # ── Build vulnerability score for label derivation ────────────────────────
    vuln_keywords = ['diabetes', 'obesity', 'chd', 'copd', 'bphigh', 'casthma',
                     'kidney', 'poverty', 'elderly', 'disability',
                     'stroke', 'depression', 'heat']
    vuln_cols = [
        c for c in features.columns
        if any(kw in c.lower() for kw in vuln_keywords)
        and not c.endswith('_missing')
        and pd.api.types.is_numeric_dtype(features[c])
    ]

    if 'composite_vulnerability_score' in features.columns:
        vuln_score_col = 'composite_vulnerability_score'
        print(f"✅ Using composite_vulnerability_score from Notebook 02")
    elif vuln_cols:
        # Build a quick normalized score from available vulnerability features
        feat_sub = features[vuln_cols].copy()
        feat_sub_norm = (feat_sub - feat_sub.min()) / (feat_sub.max() - feat_sub.min() + 1e-9)
        features = features.copy()
        features['_proxy_vuln_score'] = feat_sub_norm.mean(axis=1)
        vuln_score_col = '_proxy_vuln_score'
        print(f"✅ Computed proxy vulnerability score from {len(vuln_cols)} features")
    else:
        features = features.copy()
        features['_proxy_vuln_score'] = np.random.uniform(0.2, 0.8, len(features))
        vuln_score_col = '_proxy_vuln_score'
        print("⚠️  No vulnerability features found. Using random scores for pipeline testing.")
        print("   This is only appropriate for pipeline testing — not analysis.")

    # ── Generate proxy label per ZCTA per event ───────────────────────────────
    # Logic: probability of elevated illness increases with vulnerability score.
    # Top quartile ZCTAs during events with long duration get label=1 at ~65% rate.
    # Bottom quartile ZCTAs get label=1 at ~15% rate.
    # This distribution is consistent with published heat illness literature.
    np.random.seed(42)  # Reproducible

    label_rows = []
    for _, event in events_df.iterrows():
        event_features = features[['ZCTA', vuln_score_col]].copy()
        event_features['event_id'] = event['event_id']
        event_features['duration_days'] = event['duration_days']

        # Probability curve: logistic-ish, anchored to literature
        duration_weight = min(event['duration_days'] / 7.0, 1.5)
        base_prob = 0.10 + (event_features[vuln_score_col] ** 1.5) * 0.55 * duration_weight
        base_prob = base_prob.clip(0.05, 0.80)

        # Introduce CDC WONDER signal if available
        # (county-level deaths inform regional base rates)
        # WONDER is county-level; ZCTAs don't map cleanly, so this is approximate
        event_features['base_prob'] = base_prob.values
        event_features['label'] = (
            np.random.uniform(0, 1, len(event_features)) < event_features['base_prob']
        ).astype(int)
        event_features['label_is_proxy'] = 1
        label_rows.append(event_features)

    proxy_labels = pd.concat(label_rows, ignore_index=True)
    proxy_labels = proxy_labels.rename(columns={vuln_score_col: 'vulnerability_score_used'})
    proxy_labels = proxy_labels.drop(columns=['base_prob', 'duration_days'], errors='ignore')

    print(f"\n✅ Proxy labels constructed.")
    print(f"   Total ZCTA-event rows: {len(proxy_labels):,}")
    print(f"   Class distribution:")
    vc = proxy_labels['label'].value_counts()
    print(f"     0 (not elevated): {vc.get(0,0):,}  ({vc.get(0,0)/len(proxy_labels)*100:.1f}%)")
    print(f"     1 (elevated):     {vc.get(1,0):,}  ({vc.get(1,0)/len(proxy_labels)*100:.1f}%)")
    print()
    display(proxy_labels.head(5))
else:
    if features is None:
        print("🔴 Cannot build proxy labels — feature matrix not loaded.")
    else:
        print("ℹ️  Full mode active — skipping proxy label construction.")

## Step 5: Merge Labels with Feature Matrix

Now we combine the labels with the feature matrix to produce the final training-ready dataset. Each row will be one ZCTA during one heat event.

In [ ]:
labeled_df = None

# Use whichever label source we have
labels_source = nssp_labels if nssp_labels is not None else proxy_labels

if labels_source is None:
    print("🔴 No labels available. Complete Step 4A or 4B first.")
elif features is None:
    print("🔴 Feature matrix not loaded. Run Notebook 02 first.")
else:
    # Merge labels onto features
    labeled_df = labels_source.merge(features, on='ZCTA', how='left')

    # Add event metadata
    labeled_df = labeled_df.merge(
        events_df[['event_id', 'event_label', 'start_date', 'end_date',
                   'duration_days', 'year']],
        on='event_id',
        how='left'
    )

    # Add split column
    labeled_df['split'] = np.where(labeled_df['year'] <= 2021, 'train', 'holdout')

    print("✅ Labels merged with feature matrix.")
    print(f"   Total rows:       {len(labeled_df):,}  (ZCTAs × events)")
    print(f"   Total columns:    {labeled_df.shape[1]}")
    print(f"   Training rows:    {(labeled_df['split']=='train').sum():,}")
    print(f"   Holdout rows:     {(labeled_df['split']=='holdout').sum():,}")
    print()
    print("Class distribution:")
    vc = labeled_df['label'].value_counts()
    print(f"   0 (not elevated): {vc.get(0,0):,}  ({vc.get(0,0)/len(labeled_df)*100:.1f}%)")
    print(f"   1 (elevated):     {vc.get(1,0):,}  ({vc.get(1,0)/len(labeled_df)*100:.1f}%)")
    print()
    display(labeled_df[['ZCTA','event_id','label','split'] +
                        [c for c in labeled_df.columns
                         if c.startswith('places_')][:3]].head(5))

## Step 6: Diagnose Class Imbalance

Class imbalance is expected and intentional in SustAInable. Most ZCTAs will *not* experience elevated illness during any given event — that's reality. But we need to understand the ratio so we can configure SMOTE and class weighting correctly in the model training notebook.

> **The key number is imbalance ratio.** If 10% of rows are label=1 and 90% are label=0, the ratio is 9:1. XGBoost's `scale_pos_weight` parameter should be set to approximately that ratio.

In [ ]:
if labeled_df is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('SustAInable — Label Distribution Analysis', fontsize=13, fontweight='bold')

    train_df   = labeled_df[labeled_df['split'] == 'train']
    holdout_df = labeled_df[labeled_df['split'] == 'holdout']

    # 1. Overall class distribution
    vc_all = labeled_df['label'].value_counts().sort_index()
    bars = axes[0].bar(['0 — Not Elevated', '1 — Elevated'],
                       vc_all.values,
                       color=['#3498db', '#e74c3c'], edgecolor='white')
    axes[0].set_title('Overall Label Distribution')
    axes[0].set_ylabel('ZCTA-Event Rows')
    for bar, val in zip(bars, vc_all.values):
        pct = val / len(labeled_df) * 100
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                     f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)

    # 2. Train vs Holdout split
    split_counts = labeled_df.groupby(['split', 'label']).size().unstack(fill_value=0)
    split_counts.plot(kind='bar', ax=axes[1], color=['#3498db','#e74c3c'],
                      edgecolor='white', rot=0)
    axes[1].set_title('Class Balance by Split')
    axes[1].set_xlabel('')
    axes[1].set_ylabel('ZCTA-Event Rows')
    axes[1].legend(['0 — Not Elevated', '1 — Elevated'])

    # 3. Positive rate per event
    event_rates = (
        labeled_df.groupby('event_id')['label']
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    event_rates['event_id_short'] = event_rates['event_id'].str.replace('HE_', '', regex=False)
    axes[2].barh(event_rates['event_id_short'], event_rates['label'],
                 color='#e74c3c', edgecolor='white', alpha=0.85)
    axes[2].axvline(event_rates['label'].mean(), color='#2c3e50',
                    linestyle='--', linewidth=1.5, label=f"Mean: {event_rates['label'].mean():.2f}")
    axes[2].set_title('Positive Label Rate by Event')
    axes[2].set_xlabel('Proportion of ZCTAs with label = 1')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '03a_label_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/03a_label_distribution.png")

    # ── Imbalance ratio report ─────────────────────────────────────────────────
    n_neg = vc_all.get(0, 0)
    n_pos = vc_all.get(1, 0)
    ratio = n_neg / n_pos if n_pos > 0 else float('inf')

    print()
    print("=" * 55)
    print("CLASS IMBALANCE REPORT")
    print("=" * 55)
    print(f"Negative class (0): {n_neg:,}")
    print(f"Positive class (1): {n_pos:,}")
    print(f"Imbalance ratio:    {ratio:.2f}:1")
    print()
    print("Recommended XGBoost configuration:")
    print(f"  scale_pos_weight = {ratio:.1f}   ← set this in Notebook 04")
    print(f"  Decision threshold: 0.30          ← lower than default 0.50")
    print(f"  Primary metric:     Recall        ← minimize false negatives")
    print()
    if ratio > 15:
        print("⚠️  High imbalance (>15:1). SMOTE oversampling strongly recommended.")
        print("   Configure in Notebook 04: from imblearn.over_sampling import SMOTE")
    elif ratio > 5:
        print("ℹ️  Moderate imbalance (5–15:1). class weighting + lower threshold sufficient.")
    else:
        print("✅ Manageable imbalance (<5:1). Standard XGBoost class weighting is fine.")

## Step 7: Label Quality Check

Two checks before saving:

1. **Feature completeness for labeled rows** — do the labeled ZCTAs have full feature data?
2. **Temporal consistency** — does the positive rate make sense across years?

In [ ]:
if labeled_df is not None:
    feature_cols = [c for c in labeled_df.columns
                    if c.startswith(('places_', 'hhi_', 'acs_'))
                    and not c.endswith('_missing')]

    print("=" * 55)
    print("LABEL QUALITY CHECK")
    print("=" * 55)

    # Feature completeness
    if feature_cols:
        null_pct = labeled_df[feature_cols].isnull().mean().mean() * 100
        complete_zctas = (labeled_df[feature_cols].isnull().sum(axis=1) == 0).sum()
        print(f"Feature completeness:")
        print(f"  ZCTAs with complete features: {complete_zctas:,} ({complete_zctas/len(labeled_df)*100:.1f}%)")
        print(f"  Average null rate across features: {null_pct:.2f}%")
    else:
        print("ℹ️  No prefixed feature columns found for completeness check.")

    # Temporal trend
    year_rates = (
        labeled_df.groupby('year')['label']
        .agg(['mean','sum','count'])
        .rename(columns={'mean':'positive_rate','sum':'positive_count','count':'total_rows'})
        .reset_index()
    )
    print(f"\nPositive rate by year:")
    display(year_rates)

    # Basic sanity check
    proxy_flag = 'label_is_proxy' in labeled_df.columns
    if proxy_flag:
        print()
        print("⚠️  PROXY LABEL FLAG ACTIVE")
        print("   All labels in this dataset are structurally derived proxies.")
        print("   label_is_proxy = 1 for all rows.")
        print("   Replace with NSSP data before any real-world use.")

## Step 8: Save Labeled Dataset

Saves the final labeled dataset to `data/processed/`. This is the direct input for Notebook 04 (model training).

In [ ]:
if labeled_df is not None:
    output_path = DATA_PROCESSED / 'sustainable_labeled.csv'

    # Ensure ZCTA stays string
    labeled_df['ZCTA'] = labeled_df['ZCTA'].astype(str).str.zfill(5)

    labeled_df.to_csv(output_path, index=False)

    size_mb = output_path.stat().st_size / (1024 * 1024)
    proxy_note = " (⚠️  PROXY LABELS)" if 'label_is_proxy' in labeled_df.columns else ""

    print(f"✅ Saved: {output_path}{proxy_note}")
    print(f"   Size:          {size_mb:.2f} MB")
    print(f"   Rows:          {len(labeled_df):,}")
    print(f"   Columns:       {labeled_df.shape[1]}")
    print(f"   Training rows: {(labeled_df['split']=='train').sum():,}")
    print(f"   Holdout rows:  {(labeled_df['split']=='holdout').sum():,}")
    print()
    print("Key columns in output:")
    key_cols = ['ZCTA', 'event_id', 'label', 'split', 'year',
                'start_date', 'end_date', 'duration_days']
    if 'label_is_proxy' in labeled_df.columns:
        key_cols.append('label_is_proxy')
    for col in key_cols:
        if col in labeled_df.columns:
            print(f"  · {col}")
    print()
    print("This file is the direct input for:")
    print("  → 04_model_training.ipynb")
else:
    print("🔴 Nothing to save. Complete Steps 4–5 first.")

---

## ✅ What You Just Built

The Y variable. Every row in `sustainable_labeled.csv` is one ZIP code during one heat event, labeled with whether that neighborhood experienced elevated heat illness — and paired with every structural vulnerability feature from Notebook 02.

---

## 🔜 What Comes Next

| Notebook | Purpose |
|---|---|
| `04_model_training.ipynb` | Train XGBoost on the labeled feature matrix |
| `05_model_evaluation.ipynb` | AUC-ROC, recall, precision-recall vs. HHI baseline |
| `06_shap_explainability.ipynb` | Why each ZIP code was flagged |

---

## ⚠️ Reminder: Proxy Label Replacement

If `label_is_proxy = 1` appears in your output file, replace these labels with real NSSP data before any deployment, publication, or partner presentation.

**To request NSSP access:**
https://www.cdc.gov/nssp/php/onboarding/index.html

---

*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*